# 원신 캐릭터 정보 RAG — 1단계 평가 (코랩용)

**하는 일**: 캐릭터 124명의 **원소·무기·성급·지역** 정보로 질문과 정답을 자동으로 만들고,
오픈소스 LLM의 답을 **F1 점수**로 채점해서 **결과를 JSON으로 저장**한다. 대시보드는 이 JSON만 읽는다.

**질문 종류 (조건부 워크플로가 질문마다 경로를 고른다)**
- `lookup` : 캐릭터 한 명의 정보 ("알베도의 원소는 뭐야?") → 그 캐릭터 문서만 검색
- `filter` : 조건에 맞는 캐릭터 목록 ("몬드 출신 5성 캐릭터를 모두 알려줘") → 여러 문서를 키워드 점수로 검색
- `unknown`: 데이터에 없는 캐릭터 ("가짜이름의 원소는 뭐야?") → 근거 없음, 빈 배열 []이 정답

**비교하는 방법 4가지**: baseline(질문만) → prompt_v2(규칙+예시) → rag(+근거) → rag_v3(+근거가 붙은 예시)

**사용법**
1. 팀 드라이브 폴더 `Genshin Build Alchemist (팀)`을 우클릭 → 정리 → 바로가기 추가 → 내 드라이브 (처음 한 번만)
2. 코랩 왼쪽 열쇠 아이콘(보안 비밀)에 `GITHUB_TOKEN` 추가하고 노트북 액세스 켜기 (저장소가 비공개라 코드를 받을 때 필요, 처음 한 번만)
3. `런타임 > 런타임 유형 변경 > T4 GPU` 선택
4. 위에서부터 차례로 실행. 셀 1이 저장소 코드(core/, data/)를 받아 오고, 결과는 드라이브 `results/`에 저장한다
5. 직접 만든 질문은 드라이브 `data/human_questions.csv`에 두면 셀 12에서 평가한다


In [ ]:
# 1. 설정: 저장소 코드(core/) 받기 + 드라이브 연결 + 데이터 불러오기
#    라우터·검색은 저장소의 core/ 코드를 그대로 쓴다 (노트북 평가와 실제 챗봇이 같은 코드로 동작하게)
import hashlib
import json
import os
import random
import re
import subprocess
import sys
import time
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

REPO = "genshin-build-alchemist/genshin-build-alchemist"
REPO_BRANCH = "main"       # PR 브랜치를 시험할 때만 바꾼다 (예: "rag/router-tests")
IN_COLAB = "google.colab" in sys.modules


def github_token():
    """코랩 보안 비밀(왼쪽 열쇠 아이콘)의 GITHUB_TOKEN. 저장소가 공개면 없어도 된다."""
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return os.environ.get("GITHUB_TOKEN")


def run_git(args, token):
    """git 실행. 출력에 토큰이 찍히지 않게 가린다."""
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token, "***") if token else r.stdout + r.stderr
    if r.returncode != 0:
        raise RuntimeError("git 실패:\n" + out + "\n저장소가 비공개면 코랩 보안 비밀에 GITHUB_TOKEN을 넣었는지 확인하세요 "
                           "(notebooks/README.md 참고).")


if IN_COLAB:
    REPO_DIR = Path("/content/genshin-build-alchemist")
    token = github_token()
    url = f"https://{'x-access-token:' + token + '@' if token else ''}github.com/{REPO}.git"
    if (REPO_DIR / ".git").exists():
        run_git(["-C", str(REPO_DIR), "fetch", "--depth", "1", url, REPO_BRANCH], token)
        run_git(["-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], token)
    else:
        run_git(["clone", "--depth", "1", "--branch", REPO_BRANCH, url, str(REPO_DIR)], token)
        run_git(["-C", str(REPO_DIR), "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], token)
else:   # 로컬: 이 노트북이 들어 있는 저장소를 쓴다 (GBA_REPO 환경변수로 지정 가능)
    REPO_DIR = Path(os.environ.get("GBA_REPO") or next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core").is_dir()))
sys.path.insert(0, str(REPO_DIR))
commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip() or "unknown"
print(f"저장소 코드: {REPO_DIR} ({REPO_BRANCH} @ {commit})")

from core.kb import load_kb                       # noqa: E402  (저장소 경로를 넣은 뒤에 가져온다)
from core.retrieval import character_line, retrieve as core_retrieve   # noqa: E402
from core.router import Router                    # noqa: E402

# 팀 드라이브 폴더 (공유받은 폴더를 '내 드라이브'에 바로가기로 추가해 두어야 이 경로가 보인다)
TEAM_DIR = Path("/content/drive/MyDrive/Genshin Build Alchemist (팀)")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
if TEAM_DIR.exists():
    HUMAN_PATH = TEAM_DIR / "data" / "human_questions.csv"     # 사람이 직접 만든 질문 (선택)
    RESULT_DIR = TEAM_DIR / "results"                         # 실험 결과 JSON 폴더 (대시보드가 읽는 곳)
    print(f"팀 드라이브 사용: {TEAM_DIR}")
else:
    HUMAN_PATH = Path("human_questions.csv")
    RESULT_DIR = Path("results")
    print("팀 드라이브 폴더가 없어서 현재 폴더를 씁니다. (결과가 세션 종료 시 사라지니 주의)")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_VERSION = "info_v2"          # 질문 생성 규칙을 바꾸면 이 이름도 바꾼다 (점수 비교 기준이 달라지므로)
SEED = 42
CODE_VERSION = f"{REPO_BRANCH}@{commit}"   # 결과 JSON의 config에 남긴다 (어느 코드로 돌렸는지)
# 같은 experiment_id로 다시 실행하면 드라이브의 결과 파일을 덮어쓴다. 남기고 싶으면 ID를 바꾼다.

# 데이터는 저장소의 data/ 를 쓴다 (드라이브 data/ 는 같은 파일의 사본)
KB = load_kb(REPO_DIR / "data")
with open(REPO_DIR / "data" / "characters_ko.json", encoding="utf-8") as f:
    raw_data = json.load(f)
characters = KB.characters
CHAR_BY_NAME = KB.char_by_name
print(f"캐릭터 {len(characters)}명 불러옴 (출처: {raw_data['source']}, {raw_data['commit']})")
for field in ("element", "weapon", "rarity", "region"):
    missing = [c["name"] for c in characters if not c[field]]
    print(f"  {field:<8} 값 {len({c[field] for c in characters if c[field]})}종, 정보 없음 {len(missing)}명")


In [ ]:
# 2. 질문 + 정답(gold) 자동 생성
FIELD_KO = {"element": "원소", "weapon": "무기", "rarity": "성급", "region": "지역"}


def has_batchim(word):
    """마지막 글자에 받침이 있는지 (은/는, 이/가 고르기용). 한글이 아니면 받침 없음으로 본다."""
    ch = word[-1]
    if "가" <= ch <= "힣":
        return (ord(ch) - ord("가")) % 28 != 0
    return False


def eun(word):
    return word + ("은" if has_batchim(word) else "는")


LOOKUP_TEMPLATES = {
    "element": lambda n: f"{n}의 원소는 뭐야?",
    "weapon": lambda n: f"{eun(n)} 어떤 무기를 써?",
    "rarity": lambda n: f"{eun(n)} 몇 성 캐릭터야?",
    "region": lambda n: f"{n}의 출신 지역은 어디야?",
}

# 조건 문구: 질문 안에서 이 문구를 찾아 조건을 읽는다 (검색도 같은 문구를 쓴다)
CONDITION_PHRASE = {
    "region": lambda v: f"{v} 출신",
    "weapon": lambda v: f"{v}을 쓰는" if has_batchim(v) else f"{v}를 쓰는",
    "rarity": lambda v: v,
    "element": lambda v: f"{v} 원소",
}
PHRASE_ORDER = ["region", "weapon", "rarity", "element"]   # 자연스러운 한국어 순서
FILTER_PAIRS = [("element", "rarity"), ("weapon", "rarity"), ("element", "weapon"),
                ("region", "rarity"), ("region", "element")]
MAX_FILTER_GOLD = 10   # 정답이 너무 긴 목록은 작은 모델이 끝까지 못 쓰므로 뺀다

# 데이터에 없는 가짜 캐릭터 이름 (환각 확인용: 정답은 빈 배열)
FAKE_NAMES = ["아르셀리아", "카덴로우", "미르하젤", "토바린", "엘렌시아", "루카르도", "세이렌하", "벨트라움",
              "오르페아", "키리안테", "나디르", "솔베른", "하이델린", "마르코사", "티엔류", "이스벨라"]


def filter_question(conds):
    words = [CONDITION_PHRASE[f](conds[f]) for f in PHRASE_ORDER if f in conds]
    return f"{' '.join(words)} 캐릭터를 모두 알려줘"


def stable_split(key, eval_ratio=0.2):
    """이름 등 문자열로 train/eval을 정한다. 실행할 때마다, 컴퓨터가 달라도 같은 결과."""
    h = int(hashlib.md5(f"{SEED}-{key}".encode("utf-8")).hexdigest(), 16)
    return "eval" if (h % 1000) / 1000 < eval_ratio else "train"


# 캐릭터 단위 분리. 학습용 질문(예시, 파인튜닝 데이터)에는 평가용 캐릭터가 정답에도 근거에도 나오지 않게 한다.
CHAR_SPLIT = {c["name"]: stable_split("char-" + c["name"]) for c in characters}
TRAIN_POOL = [c for c in characters if CHAR_SPLIT[c["name"]] == "train"]


def pool_for(q):
    """질문이 볼 수 있는 캐릭터 범위: 학습용 질문은 학습용 캐릭터만, 평가/사람 질문은 전체."""
    return TRAIN_POOL if q["split"] == "train" else characters


def build_questions(characters):
    questions = []
    # (1) lookup: 캐릭터 단위로 train/eval을 나눈다 (같은 캐릭터가 양쪽에 섞이면 파인튜닝이 답을 외움)
    for c in characters:
        split = CHAR_SPLIT[c["name"]]
        for field, tmpl in LOOKUP_TEMPLATES.items():
            if not c[field]:
                continue   # 정보 없는 항목은 질문을 만들지 않는다
            questions.append({"id": f"lookup_{field}_{c['key']}", "type": "lookup", "field": field,
                              "question": tmpl(c["name"]), "gold": [c[field]], "split": split,
                              "character": c["name"]})
    # (2) filter: 두 조건 조합. 조합 단위로 train/eval을 나눈다.
    #     학습용 조합의 정답은 '학습용 캐릭터 중에서' 고른다 (평가용 캐릭터 정보가 새지 않게)
    for f1, f2 in FILTER_PAIRS:
        values1 = sorted({c[f1] for c in characters if c[f1]})
        values2 = sorted({c[f2] for c in characters if c[f2]})
        for v1 in values1:
            for v2 in values2:
                qid = f"filter_{f1}-{v1}_{f2}-{v2}"
                split = stable_split(qid)
                pool = TRAIN_POOL if split == "train" else characters
                gold = sorted(c["name"] for c in pool if c[f1] == v1 and c[f2] == v2)
                if not gold or len(gold) > MAX_FILTER_GOLD:
                    continue
                conds = {f1: v1, f2: v2}
                questions.append({"id": qid, "type": "filter", "field": f"{f1}+{f2}",
                                  "question": filter_question(conds), "gold": gold,
                                  "split": split, "conditions": conds})
    # (3) unknown: 데이터에 없는 캐릭터. 앞 절반은 train, 뒤 절반은 eval
    for i, name in enumerate(FAKE_NAMES):
        field = list(LOOKUP_TEMPLATES)[i % 4]
        questions.append({"id": f"unknown_{i:02d}", "type": "unknown", "field": field,
                          "question": LOOKUP_TEMPLATES[field](name), "gold": [],
                          "split": "train" if i < len(FAKE_NAMES) // 2 else "eval", "character": name})
    return questions


# ── 평가셋 고정 ──────────────────────────────────────────────
# 고정본은 저장소 data/eval/questions_info_v2.json 이다 (셀 1에서 받아 온 저장소에 들어 있다).
# 생성 코드나 데이터가 바뀌어 질문이 달라지면 지문(sha256)이 달라지므로 바로 알 수 있다.
# 질문 생성 규칙을 바꾸려면: DATASET_VERSION을 올리고(예: info_v3) 새 지문을 아래와 data/eval/README.md에 기록한다.
# 새 버전 고정본이 아직 저장소에 없으면 여기서 만들어 드라이브 data/에 저장한다 -> 그 파일을 PR로 저장소에 올린다.
EXPECTED_SHA256 = "5677082a29dca568cb6222c092061ff0c27983c7cc3d8a03fcd0901736c61ad7"
QUESTIONS_PATH = REPO_DIR / "data" / "eval" / f"questions_{DATASET_VERSION}.json"
NEW_FROZEN_PATH = (TEAM_DIR / "data" if TEAM_DIR.exists() else Path(".")) / f"questions_{DATASET_VERSION}.json"


def questions_sha256(qs):
    """질문 목록의 지문. 키 순서·공백과 상관없이 내용이 같으면 같은 값이 나온다."""
    text = json.dumps(qs, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def frozen_payload(qs):
    """고정 파일 내용. 만든 시각을 넣지 않아서 누가 만들어도 파일이 똑같다."""
    count = Counter((q["split"], q["type"]) for q in qs)
    return {
        "dataset_version": DATASET_VERSION,
        "sha256": questions_sha256(qs),
        "data_source": f"{raw_data['source']} {raw_data['commit']}",
        "counts": {f"{sp}_{t}": count[(sp, t)] for sp in ("train", "eval") for t in ("lookup", "filter", "unknown")},
        "eval_characters": sorted(n for n, sp in CHAR_SPLIT.items() if sp == "eval"),
        "questions": qs,
    }


generated = build_questions(characters)
if QUESTIONS_PATH.exists():
    questions = json.loads(QUESTIONS_PATH.read_text(encoding="utf-8"))["questions"]
    sha = questions_sha256(questions)
    assert sha == EXPECTED_SHA256, (f"고정 평가셋 파일 내용이 바뀌었습니다 (지문 {sha[:12]}). "
                                    "파일을 되돌리거나, 의도한 변경이면 DATASET_VERSION을 올리세요.")
    if questions_sha256(generated) != sha:
        print("⚠️ 지금 코드로 만든 질문이 고정본과 다릅니다. 평가는 고정본으로 합니다. "
              "생성 규칙을 일부러 바꾼 거라면 DATASET_VERSION을 올리세요.")
    print(f"고정 평가셋 사용: {QUESTIONS_PATH} (지문 {sha[:12]})")
else:
    questions = generated
    sha = questions_sha256(questions)
    assert sha == EXPECTED_SHA256, (f"새로 만든 질문이 {DATASET_VERSION} 고정본과 다릅니다 (지문 {sha[:12]}). "
                                    "데이터나 생성 규칙이 바뀌었으면 DATASET_VERSION을 올리고 지문을 새로 기록하세요.")
    NEW_FROZEN_PATH.write_text(json.dumps(frozen_payload(questions), ensure_ascii=False, indent=1) + "\n",
                               encoding="utf-8")
    print(f"저장소에 고정본이 없어 새로 만들었습니다: {NEW_FROZEN_PATH} (지문 {sha[:12]})\n"
          f"  -> 이 파일을 저장소 data/eval/ 에 PR로 올려 주세요.")
Q_BY_ID = {q["id"]: q for q in questions}
count = Counter((q["split"], q["type"]) for q in questions)
print(f"질문 {len(questions)}개")
for split in ("train", "eval"):
    print(f"  {split:<5} " + ", ".join(f"{t} {count[(split, t)]}" for t in ("lookup", "filter", "unknown")))
eval_chars = sorted({q["character"] for q in questions if q["type"] == "lookup" and q["split"] == "eval"})
print(f"평가용 캐릭터 {len(eval_chars)}명 (학습용 질문에는 나오지 않음)")
for qtype in ("lookup", "filter", "unknown"):
    q = next(q for q in questions if q["type"] == qtype)
    print(f"  예) [{q['type']}] {q['question']} -> {q['gold']}")

In [ ]:
# 3. 채점: 답 정규화 + 집합 F1 + 모델 답변 파싱
def canonical(text):
    """표기 차이만 맞춘다: '불 원소'->'불', '5'/'5성급'/'★5'->'5성', '몬드 출신'->'몬드'."""
    t = str(text).strip().strip("\"'「」").strip()
    t = re.sub(r"\s*(원소|출신|지역)$", "", t)
    m = re.fullmatch(r"[★☆]?\s*(\d)\s*(성급|성|star)?", t, re.I)
    if m:
        return f"{m.group(1)}성"
    return t.lower()


def set_f1(pred, gold):
    """정밀도/재현율/F1. 정답과 예측이 모두 비어 있으면 만점 (unknown 질문에서 '모른다'고 답한 경우)."""
    p, g = {canonical(x) for x in pred if str(x).strip()}, {canonical(x) for x in gold}
    tp, fp, fn = len(p & g), len(p - g), len(g - p)
    if not p and not g:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0, "tp": 0, "fp": 0, "fn": 0}
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}


def parse_list_answer(text):
    """모델 글에서 목록을 뽑는다. JSON 배열이면 그대로, 아니면 쉼표/줄바꿈으로 나눈다."""
    m = re.search(r"\[.*?\]", text, re.S)
    if m:
        try:
            return [str(x).strip() for x in json.loads(m.group(0))]
        except json.JSONDecodeError:
            pass
    parts = re.split(r"[,\n、]", text)
    return [re.sub(r"^[\-\*\d\.\s]+", "", p).strip() for p in parts if p.strip()]


def summarize(per_question):
    """문항별 결과 -> macro/micro 지표."""
    n = len(per_question)
    tp = sum(x["tp"] for x in per_question)
    fp = sum(x["fp"] for x in per_question)
    fn = sum(x["fn"] for x in per_question)
    micro_p = tp / (tp + fp) if tp + fp else 0.0
    micro_r = tp / (tp + fn) if tp + fn else 0.0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if micro_p + micro_r else 0.0
    return {"n": n,
            "macro_f1": round(sum(x["f1"] for x in per_question) / n, 4) if n else 0.0,
            "micro_precision": round(micro_p, 4), "micro_recall": round(micro_r, 4),
            "micro_f1": round(micro_f1, 4),
            "empty_answers": sum(1 for x in per_question if not x["pred"]),
            # 응답 시간 (예전 결과 파일에는 seconds가 없어서 0으로 본다)
            "avg_seconds": round(sum(x.get("seconds", 0) for x in per_question) / n, 3) if n else 0.0,
            "max_seconds": round(max((x.get("seconds", 0) for x in per_question), default=0), 3)}


def evaluate(answer_fn, qs, progress=True):
    """answer_fn(q) -> 모델 원문(문자열). 파싱과 채점은 여기서 한다. 원문도 저장한다."""
    iterator = qs
    if progress:
        try:
            from tqdm.auto import tqdm
            iterator = tqdm(qs, desc="질문 처리")
        except ImportError:
            pass
    per_question = []
    for q in iterator:
        started = time.perf_counter()
        raw = answer_fn(q)
        seconds = time.perf_counter() - started          # 질문별 응답 시간 (가이드 기본 3-2)
        pred = parse_list_answer(raw)
        s = set_f1(pred, q["gold"])
        path, docs = retrieve(q["question"], pool_for(q))
        per_question.append({"id": q["id"], "type": q["type"], "field": q["field"],
                             "question": q["question"], "gold": q["gold"], "pred": pred, "raw": raw,
                             "precision": round(s["precision"], 4), "recall": round(s["recall"], 4),
                             "f1": round(s["f1"], 4), "tp": s["tp"], "fp": s["fp"], "fn": s["fn"],
                             "seconds": round(seconds, 3),
                             "route": path,
                             "retrieved": [c["name"] for c in docs]})
    return per_question, summarize(per_question)


def breakdown(per_question, key="type"):
    """유형(type) 또는 항목(field)별 지표."""
    groups = defaultdict(list)
    for x in per_question:
        groups[x[key]].append(x)
    return {k: summarize(v) for k, v in sorted(groups.items())}


def run_self_tests():
    assert canonical("불 원소") == "불" and canonical("5") == "5성" and canonical("★4") == "4성"
    assert canonical("5성급") == "5성" and canonical("몬드 출신") == "몬드"
    assert set_f1(["알베도"], ["알베도"])["f1"] == 1.0
    assert set_f1([], [])["f1"] == 1.0                       # unknown 질문에서 빈 답 = 정답
    assert set_f1(["바위"], [])["f1"] == 0.0                 # unknown 질문에서 지어낸 답 = 오답
    assert round(set_f1(["a", "b"], ["a", "c"])["f1"], 2) == 0.5
    assert parse_list_answer('답: ["불", "물"]') == ["불", "물"]
    assert parse_list_answer("불, 물") == ["불", "물"]
    assert parse_list_answer("[]") == []
    # 정답 유출 검사: 평가용 캐릭터의 lookup 질문이 학습용에 없어야 한다
    train_chars = {q["character"] for q in questions if q["type"] == "lookup" and q["split"] == "train"}
    assert not (train_chars & set(eval_chars)), "같은 캐릭터가 학습/평가 양쪽에 있음"
    assert len({q["id"] for q in questions}) == len(questions), "질문 id 중복"
    assert not any(n in CHAR_BY_NAME for n in FAKE_NAMES), "가짜 이름이 실제 캐릭터와 겹침"
    print("채점/파싱/분리 테스트 통과")


run_self_tests()

In [ ]:
# 4. 조건부 워크플로: 저장소 core/ 의 라우터·검색을 그대로 쓴다
#    - 캐릭터 이름이 있으면 lookup  -> 그 캐릭터 문서 1개
#    - 조건 문구가 있으면 filter    -> 키워드 점수로 상위 문서 최대 30개 (모델이 조건을 직접 걸러야 함)
#    - 둘 다 없으면 unknown         -> 근거 없음
#    (챗봇에는 patch·build 경로도 있지만 이 1단계 평가셋에는 나오지 않는다)
#    라우터·검색을 고치려면 노트북이 아니라 core/router.py, core/retrieval.py 를 고쳐서 PR로 올린다.
ROUTER = Router(KB)


def find_character(text):
    return ROUTER.find_character(text)


def find_conditions(text):
    return ROUTER.find_conditions(text)


def route(text):
    return ROUTER.route(text).route


def retrieve(text, pool=None):
    """질문 -> (경로, 근거 캐릭터 목록). pool: 검색 대상 캐릭터 (기본은 전체)."""
    rr = ROUTER.route(text)
    evidence = core_retrieve(rr, KB, pool)
    return rr.route, [e["data"] for e in evidence if e["kind"] == "character"]


def doc_line(c):
    """캐릭터 문서 1줄. (1차 실험 결과: 대괄호 없는 문장형이 작은 모델에게 읽기 쉬웠다)"""
    return "- " + character_line(c)


def context_for(q):
    _, docs = retrieve(q["question"], pool_for(q))
    return "\n".join(doc_line(c) for c in docs) or "(관련 근거 없음)"


In [ ]:
# 5. 가짜 모델 2개로 파이프라인 점검 (진짜 LLM 연결 전에 저장/비교가 도는지 확인)
def oracle_reader(q):
    """근거를 완벽하게 읽는 가짜 모델 = RAG로 낼 수 있는 점수 상한. 검색이 정답을 놓치면 여기서 드러난다."""
    path, docs = retrieve(q["question"], pool_for(q))
    if path == "lookup":
        value = docs[0][q["field"]] if docs else None
        return json.dumps([value] if value else [], ensure_ascii=False)
    if path == "filter":
        conds = find_conditions(q["question"])
        names = [c["name"] for c in docs if all(c[f] == v for f, v in conds.items())]
        return json.dumps(names, ensure_ascii=False)
    return "[]"


def random_guesser(q):
    """아무거나 찍는 가짜 모델 = 바닥 점수."""
    rnd = random.Random("rand-" + q["id"])
    if q["type"] == "filter":
        return json.dumps(rnd.sample(list(CHAR_BY_NAME), 3), ensure_ascii=False)
    pool = sorted({c[q["field"]] for c in characters if c[q["field"]]})
    return json.dumps([rnd.choice(pool)], ensure_ascii=False)


def run_experiment(experiment_id, method, model_name, answer_fn, qs, notes="", config=None,
                   dataset_version=DATASET_VERSION, limit=None):
    if limit:
        qs = qs[:limit]
    start = time.time()
    per_question, metrics = evaluate(answer_fn, qs)
    result = {
        "experiment_id": experiment_id,
        "method": method,                       # baseline / prompt_v2 / rag / rag_v3 / lora ...
        "model": model_name,
        "dataset_version": dataset_version,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "seconds": round(time.time() - start, 1),
        "limit": limit,                         # None이면 전체 문항 결과
        "config": {"code_version": CODE_VERSION, **(config or {})},
        "metrics": metrics,
        "by_type": breakdown(per_question, "type"),
        "by_field": breakdown(per_question, "field"),
        "per_question": per_question,
        "notes": notes,
    }
    path = RESULT_DIR / f"{experiment_id}.json"
    path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"저장: {path}  (macro F1 = {metrics['macro_f1']}, {metrics['n']}문항, {result['seconds']}초)")
    return result


def load_results(prefix="", result_dir=RESULT_DIR):
    paths = Path(result_dir).glob(f"{prefix}*.json")
    return sorted((json.loads(p.read_text(encoding="utf-8")) for p in paths),
                  key=lambda r: (r["created_at"], r["experiment_id"]))


def print_comparison(results):
    print(f"{'실험':<20}{'방법':<11}{'macro F1':>9}{'micro P':>9}{'micro R':>9}{'micro F1':>10}{'빈답':>6}{'평균초':>8}")
    for r in results:
        m = r["metrics"]
        print(f"{r['experiment_id']:<20}{r['method']:<11}{m['macro_f1']:>9.3f}{m['micro_precision']:>9.3f}"
              f"{m['micro_recall']:>9.3f}{m['micro_f1']:>10.3f}{m.get('empty_answers', 0):>6}"
              f"{m.get('avg_seconds', 0):>8.2f}")
    if results:
        types = sorted({t for r in results for t in r.get("by_type", {})})
        print(f"\n{'유형별 macro F1':<20}" + "".join(f"{t:>10}" for t in types))
        for r in results:
            row = f"{r['experiment_id']:<20}"
            for t in types:
                m = r.get("by_type", {}).get(t)
                row += f"{m['macro_f1']:>10.3f}" if m else f"{'-':>10}"
            print(row)


# 평가 질문 순서를 섞어 둔다 (LIMIT으로 앞부분만 돌려도 lookup/filter/unknown이 고루 들어가게)
EVAL_QS = sorted((q for q in questions if q["split"] == "eval"),
                 key=lambda q: hashlib.md5(q["id"].encode("utf-8")).hexdigest())
demo_a = run_experiment("demo01_random", "random", "fake", random_guesser, EVAL_QS, "찍기 = 바닥 점수")
demo_b = run_experiment("demo02_oracle", "oracle", "fake", oracle_reader, EVAL_QS, "근거 완벽 독해 = RAG 상한")
assert demo_b["metrics"]["macro_f1"] == 1.0, "검색이 정답을 놓치는 질문이 있음 -> retrieve() 확인 필요"
print()
print_comparison(load_results("demo"))

---
# 진짜 오픈소스 LLM 연결

**먼저**: 코랩 메뉴 `런타임 > 런타임 유형 변경 > T4 GPU`. 아래 셀 출력에 `GPU 사용 가능: True`가 나와야 한다.

In [ ]:
# 6. 모델 불러오기 (처음 한 번은 내려받느라 몇 분 걸린다. 끝나면 GPU RAM이 6GB 안팎으로 올라간다)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"   # 다른 모델로 바꿔서 비교 가능 (T4 16GB에는 3B~4B급이 무난)
MAX_NEW_TOKENS = 160                     # filter 질문은 이름이 최대 10개라 조금 넉넉하게

print("GPU 사용 가능:", torch.cuda.is_available())
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")
except TypeError:   # transformers 버전에 따라 인자 이름이 다르다
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")


def generate(prompt, max_new_tokens=MAX_NEW_TOKENS):
    """프롬프트 -> 모델 답변. 샘플링 끔(매번 같은 답), ']'가 나오면 멈춘다."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    kwargs = dict(max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    with torch.no_grad():
        try:
            out = model.generate(**inputs, **kwargs, stop_strings=["]"], tokenizer=tokenizer)
        except (TypeError, ValueError):   # 오래된 transformers는 stop_strings를 모른다
            out = model.generate(**inputs, **kwargs)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
# 7. 프롬프트 5종류
RULES = ("규칙:\n"
         "- 질문에서 묻는 값(또는 캐릭터 이름)만 JSON 배열로 출력하세요. 예: [\"A\", \"B\"]\n"
         "- 설명이나 다른 말은 쓰지 마세요.\n"
         "- 해당하는 항목을 빠짐없이 모두 적으세요.\n"
         "- 모르는 캐릭터이거나 해당하는 항목이 없으면 빈 배열 []을 출력하세요.")
RAG_NOTE = "(위 근거에 있는 내용만 사용하세요. 근거에 없는 캐릭터면 []을 출력하세요.)"
# rag_v4: DEV(rag_v3)에서 성급·지역 질문에 빈 답 []이 많아 근거 읽는 법과 조건 규칙을 더했다
RULES_V4 = RULES + (
    "\n- 근거의 '5성', '4성'이 성급이고, '지역'이 출신 지역입니다."
    "\n- 조건이 여러 개인 질문은 조건을 모두 만족하는 캐릭터만 고르세요.")


def few_shot_examples():
    """예시는 학습용(train) 질문에서만 고른다: lookup 1개, filter 1개, unknown 1개."""
    train = [q for q in questions if q["split"] == "train"]
    picked = [next(q for q in train if q["type"] == "lookup" and q["field"] == "element"),
              next(q for q in train if q["type"] == "filter" and 2 <= len(q["gold"]) <= 3),
              next(q for q in train if q["type"] == "unknown")]
    return picked


def example_block(ex, with_context):
    block = f"예시 질문: {ex['question']}\n예시 답: {json.dumps(ex['gold'], ensure_ascii=False)}"
    if with_context:
        block = f"[예시 근거]\n{context_for(ex)}\n" + block
    return block


def make_prompt(q, method):
    if method == "baseline":
        return f"{q['question']}\nJSON 배열로만 답하세요."
    parts = [RULES]
    if method == "rag_v4":
        parts = [RULES_V4]
    with_ctx_shots = method in ("rag_v3", "rag_v4")
    shots = few_shot_v4() if method == "rag_v4" else few_shot_examples()
    parts.append("\n\n".join(example_block(ex, with_ctx_shots) for ex in shots))
    if method in ("rag", "rag_v3", "rag_v4"):
        parts.append(f"[근거]\n{context_for(q)}\n{RAG_NOTE}")
    parts.append(f"[질문]\n{q['question']}\n답:")
    return "\n\n".join(parts)


def build_dev_set(n_per_field=10, n_filter=16, n_unknown=4):
    """프롬프트·설정을 고를 때만 쓰는 검증용(DEV) 질문 60개. 학습용 질문에서 뽑고 few-shot 예시는 뺀다.
    평가 115문항(EVAL_QS)과 사람 질문은 프롬프트를 고를 때 보지 않는다 (PRD 4.3).
    질문 id의 해시 순서로 뽑아서 누가 돌려도 같은 60개가 나온다."""
    shot_ids = {ex["id"] for ex in few_shot_examples()}
    pool = sorted((q for q in questions if q["split"] == "train" and q["id"] not in shot_ids),
                  key=lambda q: hashlib.md5(("dev-" + q["id"]).encode("utf-8")).hexdigest())
    dev = []
    for field in ("element", "weapon", "rarity", "region"):
        dev += [q for q in pool if q["type"] == "lookup" and q["field"] == field][:n_per_field]
    dev += [q for q in pool if q["type"] == "filter"][:n_filter]
    dev += [q for q in pool if q["type"] == "unknown"][:n_unknown]
    return sorted(dev, key=lambda q: hashlib.md5(q["id"].encode("utf-8")).hexdigest())


DEV_QS = build_dev_set()


def few_shot_v4():
    """rag_v4 예시 5개 = 기존 3개 + 성급·지역 lookup 1개씩. 학습용 질문 중 DEV와 겹치지 않는 것에서 고른다."""
    dev_ids = {q["id"] for q in DEV_QS}
    pool = [q for q in questions if q["split"] == "train" and q["id"] not in dev_ids]
    extra = [next(q for q in pool if q["type"] == "lookup" and q["field"] == f) for f in ("rarity", "region")]
    return few_shot_examples() + extra

print(f"DEV 세트 {len(DEV_QS)}문항: " + ", ".join(f"{t} {n}" for t, n in sorted(Counter(q['type'] for q in DEV_QS).items())))


METHODS = [
    ("baseline", "질문만 그대로"),
    ("prompt_v2", "규칙 + 예시 3개 (근거 없음)"),
    ("rag", "prompt_v2 + 검색한 근거"),
    ("rag_v3", "rag + 예시에도 근거를 붙임 (1차 실험에서 효과가 컸던 방식)"),
    ("rag_v4", "rag_v3 + 성급·지역 예시 추가 + 근거 읽는 법·조건 규칙"),
]


def make_answer_fn(method):
    def answer_fn(q):
        return generate(make_prompt(q, method))
    return answer_fn

In [ ]:
# 8. 실험 전 점검 (모델 호출 없음): 통과해야 실험 결과를 믿을 수 있다
def run_prompt_tests():
    eval_ids = {q["id"] for q in EVAL_QS}
    shots = few_shot_examples()
    assert all(ex["split"] == "train" for ex in shots), "예시가 학습용이 아님"
    for method, _ in METHODS:
        prompt = make_prompt(EVAL_QS[0], method)
        assert EVAL_QS[0]["question"] in prompt
    # 학습용 질문(예시·파인튜닝 데이터)의 문구/정답/근거 어디에도 평가용 캐릭터가 나오면 안 된다
    eval_set = {n for n, sp in CHAR_SPLIT.items() if sp == "eval"}
    for q in questions:
        if q["split"] != "train":
            continue
        _, docs = retrieve(q["question"], pool_for(q))
        seen = {d["name"] for d in docs} | set(q["gold"]) | {find_character(q["question"])}
        assert not (seen & eval_set), f"학습용 질문 {q['id']}에 평가용 캐릭터: {seen & eval_set}"
    # DEV 세트: 학습용에서만, few-shot 예시와 겹치지 않게, 평가용 캐릭터 없이
    dev_ids = {q["id"] for q in DEV_QS}
    assert all(q["split"] == "train" for q in DEV_QS), "DEV에 학습용이 아닌 질문이 있음"
    assert not (dev_ids & {ex["id"] for ex in shots}), "DEV와 few-shot 예시가 겹침"
    assert not (dev_ids & eval_ids), "DEV와 평가 질문이 겹침"
    assert len(DEV_QS) == 60 and len(dev_ids) == 60, f"DEV 세트 크기 이상: {len(DEV_QS)}"
    shots_v4 = few_shot_v4()
    assert len(shots_v4) == 5 and all(ex["split"] == "train" for ex in shots_v4), "rag_v4 예시 이상"
    assert not (dev_ids & {ex["id"] for ex in shots_v4}), "DEV와 rag_v4 예시가 겹침"
    # 라우터 정확도: 자동 생성 질문은 경로가 100% 맞아야 한다
    wrong = [q["id"] for q in questions if route(q["question"]) != q["type"]]
    assert not wrong, f"라우터가 틀린 질문: {wrong[:5]}"
    # 사람 말투 예시도 맞게 판별되는지
    samples = {"소 원소 뭐야?": "lookup", "진은 무슨 무기 써?": "lookup", "원소 반응이 뭐야?": "unknown",
               "몬드 5성 누구 있어?": "filter", "불원소 활 캐릭터": "filter", "5성급 바위 속성 캐릭터": "filter",
               "나타나는 캐릭터": "unknown"}
    for text, expected in samples.items():
        assert route(text) == expected, f"라우터: {text!r} -> {route(text)} (기대 {expected})"
    assert find_conditions("몬드 5성 누구 있어?") == {"region": "몬드", "rarity": "5성"}
    # 근거 유무
    assert "[근거]" not in make_prompt(EVAL_QS[0], "prompt_v2")
    assert "[근거]" in make_prompt(EVAL_QS[0], "rag")
    assert "[예시 근거]" in make_prompt(EVAL_QS[0], "rag_v3")
    assert "'5성', '4성'이 성급" in make_prompt(EVAL_QS[0], "rag_v4")
    assert "'5성', '4성'이 성급" not in make_prompt(EVAL_QS[0], "rag_v3")
    sizes = [len(retrieve(q["question"])[1]) for q in questions if q["type"] == "filter"]
    print(f"프롬프트/라우터/검색 테스트 통과 (filter 근거 문서 수: 평균 {sum(sizes) / len(sizes):.1f}, 최대 {max(sizes)})")
    print("\n[rag_v4 프롬프트 예시]\n" + make_prompt(next(q for q in EVAL_QS if q["type"] == "filter"), "rag_v4"))


run_prompt_tests()

In [ ]:
# 9. 속도 점검: 질문 1개가 몇 초 걸리는지 먼저 잰다
for method, _ in METHODS:
    q0 = next(q for q in EVAL_QS if q["type"] == "filter")
    start = time.time()
    answer = generate(make_prompt(q0, method))
    print(f"{method:<10} {time.time() - start:5.1f}초 / 프롬프트 {len(make_prompt(q0, method)):>5}자 / 답: {answer[:60]!r}")
print(f"\n평가 질문 {len(EVAL_QS)}개 x {len(METHODS)}방법 = 모델 호출 {len(EVAL_QS) * len(METHODS)}번")

In [ ]:
# 10. 진짜 모델로 5가지 방법 비교 -> results/info_*.json
LIMIT = None   # 처음에는 20문항으로 시간을 재 본다. 괜찮으면 None(전체)으로 바꿔 다시 실행
for i, (method, notes) in enumerate(METHODS):
    run_experiment(f"info{10 + i}_{method}-V2", method, MODEL_ID, make_answer_fn(method), EVAL_QS, notes,
                   limit=LIMIT,
                   config={"max_new_tokens": MAX_NEW_TOKENS, "few_shot": len(few_shot_examples()),
                           "example_prompt": make_prompt(EVAL_QS[0], method)})
print()
print_comparison(load_results("info"))

In [ ]:
# 10-1. 프롬프트 개발은 DEV 세트로 (결과: results/dev*.json)
#   - 새 프롬프트(예: rag_v4)는 셀 7의 METHODS와 make_prompt()에 추가한 뒤 여기서 DEV로 비교한다.
#   - DEV에서 고른 최종 프롬프트만 셀 10에서 평가 115문항으로 한 번 돌린다. 평가 결과를 보고 다시 고치지 않는다.
#   - 사람이 만든 질문(셀 12) 결과도 프롬프트를 고를 때 보지 않는다.
DEV_METHODS = ["baseline","prompt_v2","rag","rag_v3"]   # 비교할 방법 이름 (METHODS에 있는 것). 예: ["rag_v3", "rag_v4"]
for method in DEV_METHODS:
    notes = dict(METHODS).get(method, "")
    run_experiment(f"dev_{method}-V2", method, MODEL_ID, make_answer_fn(method), DEV_QS, notes,
                   dataset_version=f"{DATASET_VERSION}-dev",
                   config={"max_new_tokens": MAX_NEW_TOKENS, "split": "dev"})
print()
print_comparison(load_results("dev"))


In [ ]:
# 10-2. 후처리 효과: 저장된 결과의 모델 원문에 후처리만 적용해 다시 채점 (모델 호출 없음, GPU 필요 없음)
#   후처리 함수는 core/answer.py의 clean_answer (챗봇과 같은 코드). 읽기·저장 모두 results/ 폴더
from core.answer import POST_CUTOFF, clean_answer


def evidence_texts(q):
    return [line[2:] for line in context_for(q).splitlines() if line.startswith("- ")]


def save_post(experiment_id, qs, cutoff=POST_CUTOFF):
    r = json.loads((RESULT_DIR / f"{experiment_id}.json").read_text(encoding="utf-8"))
    q_by_id = {q["id"]: q for q in qs}
    rows = []
    for x in r["per_question"]:
        pred = clean_answer(parse_list_answer(x["raw"]), evidence_texts(q_by_id[x["id"]]), cutoff)
        s = set_f1(pred, x["gold"])
        rows.append({**x, "pred": pred, "precision": round(s["precision"], 4), "recall": round(s["recall"], 4),
                     "f1": round(s["f1"], 4), "tp": s["tp"], "fp": s["fp"], "fn": s["fn"]})
    result = {**r, "experiment_id": f"{experiment_id}_post", "method": f"{r['method']}+post",
              "created_at": datetime.now().isoformat(timespec="seconds"),
              "config": {**r["config"], "postprocess": "clean_answer", "cutoff": cutoff},
              "metrics": summarize(rows), "by_type": breakdown(rows, "type"), "by_field": breakdown(rows, "field"),
              "per_question": rows,
              "notes": f"{experiment_id}의 모델 원문에 후처리만 적용. 모델은 다시 돌리지 않음"}
    path = RESULT_DIR / f"{result['experiment_id']}.json"
    path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"저장: {path}  (macro F1 {r['metrics']['macro_f1']} -> {result['metrics']['macro_f1']})")


#save_post("dev_rag_v4-V2", DEV_QS)
#save_post("info14_rag_v4-V2", EVAL_QS)   # 평가: cutoff를 정한 뒤 한 번만


In [ ]:
# 11. 오답 보기: 실험 하나를 골라 틀린 문항의 정답/모델 원문을 본다
def show_failures(experiment_id, qtype=None, max_items=15):
    r = json.loads((RESULT_DIR / f"{experiment_id}.json").read_text(encoding="utf-8"))
    misses = [x for x in r["per_question"] if x["f1"] < 1.0 and (qtype is None or x["type"] == qtype)]
    print(f"[{experiment_id}] 틀린 문항 {len(misses)}개 / {len(r['per_question'])}개")
    for x in misses[:max_items]:
        print(f"- ({x['type']}/{x['field']}, F1 {x['f1']:.2f}) {x['question']}")
        print(f"    정답: {x['gold']}")
        print(f"    원문: {x['raw']!r}")


show_failures("info14_rag_v4")

In [ ]:
# 12. 사람이 만든 질문 평가 (드라이브 data/human_questions.csv가 있을 때만)
#     형식(권장): question,target,note  -> 정답은 target을 보고 데이터에서 자동으로 채운다 (core/human_questions.py)
#        target 예: '알베도/원소', '지역=몬드, 성급=5성', 빈칸(데이터에 없는 질문)
#     형식(예전): question,answer,note  -> answer는 '|'로 구분, 빈칸이면 정답 []
#     질문 종류는 라우터가 자동으로 정한다. 사람 질문은 학습/프롬프트 고르기에 절대 쓰지 않는다.
import csv

from core.human_questions import fill_rows   # noqa: E402


def load_human_questions(path=HUMAN_PATH):
    if not Path(path).exists():
        print(f"{path} 가 없어서 건너뜁니다.")
        return []
    with open(path, encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    if rows and "target" in rows[0]:
        rows, errors = fill_rows(rows, KB)
        for i, row in enumerate(rows, 2):
            if row["check"]:
                print(f"  {i}행 '{row['question']}': {row['check']}")
        if errors:
            print(f"  -> 오류 {errors}줄은 평가에서 뺍니다. target을 고쳐 주세요.")
        rows = [r for r in rows if not r["check"].startswith("오류")]
    qs = []
    for i, row in enumerate(rows, 1):
        text = (row.get("question") or "").strip()
        if not text:
            continue
        gold = [a.strip() for a in (row.get("answer") or "").split("|") if a.strip()]
        qs.append({"id": f"human_{i:03d}", "type": route(text), "field": "human", "question": text,
                   "gold": gold, "split": "human", "note": (row.get("note") or "").strip(),
                   "target": (row.get("target") or "").strip()})
    print(f"사람 질문 {len(qs)}개: " + ", ".join(f"{t} {n}" for t, n in Counter(q['type'] for q in qs).items()))
    # 라우터가 '모르는 질문'으로 봤는데 정답이 있는 경우 = 라우터가 질문을 못 알아들은 것 (규칙 보완 대상)
    for q in qs:
        if q["type"] == "unknown" and q["gold"]:
            print(f"  [라우터 확인 필요] {q['question']}  (정답 {q['gold']}인데 근거를 못 찾음)")
    return qs


HUMAN_QS = load_human_questions()
if HUMAN_QS:
    for i, (method, notes) in enumerate(METHODS):
        run_experiment(f"human{10 + i}_{method}", method, MODEL_ID, make_answer_fn(method), HUMAN_QS,
                       notes, dataset_version="human_v1")
    print()
    print_comparison(load_results("human"))

In [ ]:
# 13. 파인튜닝(LoRA) 준비: 학습용 질문만 파일로 저장 (평가/사람 질문은 넣지 않는다)
#     고정 평가셋(questions_info_v2.json)과 섞이지 않게 이름을 따로 쓴다
train_qs = [q for q in questions if q["split"] == "train"]
export = {"dataset_version": DATASET_VERSION, "source_sha256": questions_sha256(questions), "train": train_qs}
TRAIN_PATH = (TEAM_DIR / "data" if TEAM_DIR.exists() else Path(".")) / f"train_questions_{DATASET_VERSION}.json"
TRAIN_PATH.write_text(json.dumps(export, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"저장: {TRAIN_PATH} (train {len(train_qs)}개, 평가 {len(EVAL_QS)}개는 넣지 않음)")
